In [103]:
import pandas as pd
import numpy as np
import random
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import Ridge

from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt


In [118]:
df = pd.read_csv("friends_transcripts.tsv", sep="\t")

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r"[^a-zA-Z\s]", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

df["clean_text"] = df["transcript"].apply(clean_text)
df = df.sort_values(["season_id", "episode_id", "scene_id", "utterance_id"])


In [105]:
df["prev_text"] = df["clean_text"].shift(1)
df["prev_speaker"] = df["speaker"].shift(1)

pairs = df.dropna(subset=["prev_text"])
pairs = pairs[["prev_speaker", "prev_text", "speaker", "clean_text"]]

main_chars = [
    "Ross Geller", "Rachel Green", "Monica Geller",
    "Phoebe Buffay", "Joey Tribbiani", "Chandler Bing"
]

pairs = pairs[pairs["prev_speaker"].isin(main_chars)]
pairs = pairs[pairs["speaker"].isin(main_chars)]


In [106]:
pairs["model_input"] = pairs["prev_speaker"].str.lower() + " " + pairs["prev_text"]
pairs["model_output"] = pairs["clean_text"].astype(str)

vectorizer_input = TfidfVectorizer(stop_words="english", max_features=5000)
vectorizer_output = TfidfVectorizer(stop_words="english", max_features=5000)

X = vectorizer_input.fit_transform(pairs["model_input"])

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
Y_emb = embed_model.encode(pairs["model_output"].tolist())


In [ ]:
from sklearn.model_selection import train_test_split

# test/train split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y_emb, test_size=0.2, random_state=42
)

# model
reg = MultiOutputRegressor(Ridge(alpha=1.0))
reg.fit(X_train, Y_train)


MultiOutputRegressor(estimator=Ridge())

In [ ]:



# predict reply
def predict_reply(prev_speaker, prev_line, next_speaker, temperature=0.8):
    inp = f"{prev_speaker.lower()} {clean_text(prev_line)}"
    x_vec = vectorizer_input.transform([inp])
    
    y_vec = reg.predict(x_vec)[0]

    sims = cosine_similarity([y_vec], Y_emb)[0]

    # sample
    scaled = sims / temperature
    probs = np.exp(scaled) / np.sum(np.exp(scaled))

    idx = np.random.choice(len(probs), p=probs)
    
    result = pairs.iloc[idx]
    return result["speaker"], result["model_output"]


In [ ]:
text_col = "transcript"

# find topics
def detect_topic(text):
    t =str(text).lower()

    if any(w in t for w in ["love", "date", "kiss", "wedding", "relationship"]):
        return "relationships"
    if any(w in t for w in ["job", "work", "office", "boss"]):
        return "work"
    if any(w in t for w in ["food", "eat", "dinner", "restaurant"]):
        return "food"
    if any(w in t for w in ["party", "birthday", "celebrate"]):
        return "events"
    if any(w in t for w in ["apartment", "couch", "roommate"]):
        return "home"

    return "misc"

df["topic"] = df[text_col].apply(detect_topic)

# random topic
topic = np.random.choice(df["topic"].unique())

topic_df = df[df["topic"] == topic].reset_index(drop=True)

print("Chosen topic:", topic)
print(topic_df.head())


Chosen topic: home
  season_id episode_id scene_id utterance_id        speaker  \
0       s01        e01      c14         u023  Monica Geller   
1       s01        e06      c05         u011  Chandler Bing   
2       s01        e07      c03         u005  Phoebe Buffay   
3       s01        e10      c01         u011  Monica Geller   
4       s01        e10      c01         u013  Monica Geller   

                                              tokens  \
0  [['Well', ',', 'that', "'s", 'it', 'You', 'gon...   
1  [["C'mon", ',', 'we', "'re", 'roommates', '!']...   
2  [['Can', 'I', 'borrow', 'the', 'phone', '?'], ...   
3  [['Ross', ',', 'is', 'he', 'gon', 'na', 'live'...   
4  [['Why', 'do', "n't", 'you', 'just', 'get', 'a...   

                                          transcript  \
0      Well, that's it You gonna crash on the couch?   
1        C'mon, we're roommates! My eyes!! My eyes!!   
2  Can I borrow the phone? I want to call my apar...   
3  Ross, is he gonna live with you, like,

In [ ]:
from sentence_transformers import SentenceTransformer

# build embeddings for the cleaned transcript text
embed = SentenceTransformer("all-MiniLM-L6-v2")

emb = embed.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True
)
import numpy as np

# speaker-speaker frequencies
speakers = df["speaker"].unique().tolist()
S = len(speakers)
speaker_to_idx = {s:i for i,s in enumerate(speakers)}

# initialize matrix
reply_matrix = np.zeros((S, S))

# build frequencies: counts of (prev_speaker → next_speaker)
for i in range(len(df)-1):
    a = df.iloc[i]["speaker"]
    b = df.iloc[i+1]["speaker"]
    reply_matrix[speaker_to_idx[a], speaker_to_idx[b]] += 1

# convert raw counts into probability distributions
reply_probs = reply_matrix / reply_matrix.sum(axis=1, keepdims=True)
reply_probs = np.nan_to_num(reply_probs)  # toss nans

def weight_by_reply_freq(prev_speaker, candidate_indices):
    """Return normalized weights for candidates based on reply frequency."""
    prev_idx = speaker_to_idx[prev_speaker]

    # base weights
    weights = []
    for i in candidate_indices:
        next_sp = df.iloc[i]["speaker"]
        w = reply_probs[prev_idx, speaker_to_idx[next_sp]]
        weights.append(w)

    weights = np.array(weights)

    # fallback if weights are all zero
    if weights.sum() == 0:
        return np.ones(len(candidate_indices)) / len(candidate_indices)

    return weights / weights.sum()



Batches: 100%|██████████| 2106/2106 [01:09<00:00, 30.21it/s]


In [ ]:
def generate_scene(num_lines=10):
    # start line
    idx = np.random.randint(len(topic_df))
    current_line = topic_df.iloc[idx]

    scene = []
    scene.append(f"{current_line['speaker']}: {current_line['clean_text']}")

    current_emb = emb[df.index.get_loc(current_line.name)]
    prev_speaker = current_line["speaker"]

    for _ in range(num_lines - 1):

        sims = cosine_similarity([current_emb], emb)[0]

        # choose on topic replies
        top_idx = sims.argsort()[-50:][::-1]

        # get rid of replies like "yes"
        filtered = [i for i in top_idx if len(df.iloc[i]["clean_text"].split()) > 3]
        if len(filtered) == 0:
            filtered = top_idx

        # choose speaker based on frequencies
        weights = weight_by_reply_freq(prev_speaker, filtered)

        next_idx = np.random.choice(filtered, p=weights)

        next_line = df.iloc[next_idx]
        scene.append(f"{next_line['speaker']}: {next_line['clean_text']}")

        current_emb = emb[next_idx]
        prev_speaker = next_line["speaker"]

    return "\n".join(scene)


In [131]:
print(generate_scene())

Monica Geller: it s gonna be really scary i mean god when we have a baby there s gonna be so much that we re not able to control i mean the apartment s gonna be a mess i won t have time to clean it what if the baby gets into the ribbon drawer messes up all the ribbons what if there s no room for a ribbon drawer because the baby s stuff takes up all the space where will all the ribbons go
Chandler Bing: oh yeah it was great you should be a chef
Monica Geller: i do i m a professional chef oh relax it s not a courtroom drama
Ross Geller: well is this hillary your hot assistant chef hillary
Monica Geller: no no no he is totally incompetent i called the chef who recommended him to me he said ha ha gotcha
Phoebe Buffay: well maybe he was just nervous y know you can be very intimidating and besides i ve met your pastry chef and she can stand to be taken down a peg or two
Ross Geller: no no no uh he just he just really freaked me out before
Rachel Green: ohh god i just got so nervous that he w